# Hunyuan3D Avatar 3D (Colab) — Fixed

Pipeline for Colab free-tier T4 GPU:
- **Hunyuan3D-2mini** for shape generation
- **Hunyuan3D-Paint** for texture generation
- Exports `.glb` files + delivery `.zip`

> All 11 known issues from the original notebook are fixed.


## 1. GPU Validation

Hard gate: stops execution immediately if no GPU is available.


In [ ]:
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "GPU not detected. Go to Runtime > Change runtime type > "
        "select T4 GPU, then restart the runtime."
    )

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "torch.cuda.is_available() is False. "
        "Ensure GPU runtime is enabled and restart."
    )
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")


## 2. Input Image

Upload or select an input image. Output filenames derive from the image filename.


In [ ]:
from pathlib import Path
from google.colab import files

PNG_CANDIDATES = [
    Path("/content/avatar-man-1.png"),
]
PNG_CANDIDATES += sorted(Path("/content").glob("*.png"))

existing = next((p for p in PNG_CANDIDATES if p.exists()), None)
if existing is None:
    print("No PNG found in /content/. Please upload your image file...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    first_name = next(iter(uploaded.keys()))
    existing = Path("/content") / first_name

print(f"Image selected: {existing}")
SELECTED_IMAGE = str(existing)
AVATAR_NAME = existing.stem
print(f"Avatar name: {AVATAR_NAME}")


## 3. Configuration

Memory presets tuned for T4 (16 GB VRAM):
- **MAX_QUALITY**: steps=35, chunks=20000 (may OOM)
- **BALANCED**: steps=28, chunks=12000 (recommended)
- **SAFE**: steps=24, chunks=8000 (guaranteed to fit)


In [ ]:
INPUT_IMAGE = SELECTED_IMAGE
OUT_DIR = f"/content/outputs/{AVATAR_NAME}-hunyuan"
SEED = 12345
OCTREE_RESOLUTION = 320

NUM_INFERENCE_STEPS = 28
NUM_CHUNKS = 12000

print(f"Input:  {INPUT_IMAGE}")
print(f"Output: {OUT_DIR}")
print(f"Seed:   {SEED}, Steps: {NUM_INFERENCE_STEPS}, Chunks: {NUM_CHUNKS}")


## 4. HuggingFace Authentication (Optional)

Set `HF_TOKEN` in Colab Secrets if model weights become gated. Skip otherwise.


In [ ]:
import os

hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    import subprocess
    subprocess.run(
        ["huggingface-cli", "login", "--token", hf_token, "--add-to-git-credential"],
        check=False, capture_output=True,
    )
    print("HuggingFace token configured.")
else:
    print("No HF_TOKEN found. Skipping -- public models will still work.")


## 5. Install Hunyuan3D-2

Handles: auto-download from GitHub, smart torch check, `diffusers==0.31.0` pin, `pip install .` instead of deprecated `setup.py`, `CUDA_HOME` auto-detection.

Takes ~5–8 minutes on first run.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/Hunyuan3D-2")
ZIP_URL = "https://codeload.github.com/Tencent-Hunyuan/Hunyuan3D-2/zip/refs/heads/main"


def run(cmd, check=True):
    print(f"\n$ {cmd}")
    result = subprocess.run(cmd, shell=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result.returncode == 0


# Clean previous install
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
for stale_dir in (Path("/content/Hunyuan3D-2-main"), Path("/content/Hunyuan3D-2-master")):
    if stale_dir.exists():
        shutil.rmtree(stale_dir)

# Issue #1: Auto-download repo zip from GitHub
ZIP_CANDIDATES = [
    Path("/content/Hunyuan3D-2-main.zip"),
    Path("/content/Hunyuan3D-2.zip"),
]
ZIP_CANDIDATES += sorted(Path("/content").glob("*.zip"))
zip_path = next((z for z in ZIP_CANDIDATES if z.exists()), None)

if zip_path is None:
    print("Downloading Hunyuan3D-2 from GitHub...")
    zip_path = Path("/content/Hunyuan3D-2-main.zip")
    import urllib.request
    urllib.request.urlretrieve(ZIP_URL, zip_path)
    print(f"Downloaded {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

print(f"Using repo zip: {zip_path}")
run(f'unzip -q "{zip_path}" -d /content')

if not REPO_DIR.exists():
    extracted_candidates = [
        p for p in Path("/content").iterdir()
        if p.is_dir() and (p / "hy3dgen").exists() and (p / "setup.py").exists()
    ]
    if not extracted_candidates:
        raise RuntimeError("Zip extracted, but no Hunyuan3D repo root was found.")
    extracted_candidates[0].rename(REPO_DIR)

assert REPO_DIR.exists(), "Repository setup failed."

# Issue #3: Smart torch check -- skip reinstall if Colab version is compatible
import torch
torch_ok = torch.cuda.is_available() and torch.version.cuda and torch.version.cuda.startswith("12")
if torch_ok:
    print(f"Using pre-installed torch {torch.__version__} (CUDA {torch.version.cuda})")
else:
    print("Reinstalling torch for CUDA 12.1 compatibility...")
    run("python3 -m pip install --upgrade pip")
    run("python3 -m pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121")

# Install core dependencies
run("cd /content/Hunyuan3D-2 && python3 -m pip install --upgrade pip")
run("cd /content/Hunyuan3D-2 && python3 -m pip install -r requirements.txt")

# Issue #2: Pin diffusers to 0.31.0 (confirmed working with Hunyuan3D-2)
run("python3 -m pip install diffusers==0.31.0")
run("cd /content/Hunyuan3D-2 && python3 -m pip install -e .")

# Issue #5: Set CUDA_HOME before building extensions
cuda_candidates = ["/usr/local/cuda", "/usr/local/cuda-12", "/usr/local/cuda-12.1"]
for c in cuda_candidates:
    if os.path.exists(os.path.join(c, "bin", "nvcc")):
        os.environ["CUDA_HOME"] = c
        print(f"CUDA_HOME set to {c}")
        break
else:
    print("Warning: nvcc not found in standard locations.")

# Issue #4: Use pip install . instead of deprecated setup.py install
run("cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && pip install .")
run("cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && pip install .")

# Verify
import importlib
importlib.invalidate_caches()
import torch as _t
import diffusers as _d
print(f"\ntorch {_t.__version__}")
print(f"diffusers {_d.__version__}")
print("Environment ready.")


## 6. Generate 3D Avatar

1. Background removal
2. Shape generation (Hunyuan3D-2mini)
3. Texture generation (Hunyuan3D-Paint) with mesh simplification to avoid T4 OOM

Shape: ~10–15 min | Texture: ~5–10 min on T4


In [ ]:
import gc
import json
import os
import shutil
import time
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from PIL import Image

os.chdir("/content/Hunyuan3D-2")

from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

# Issue #10: Colab keepalive to prevent auto-disconnect
try:
    from IPython.display import display, Javascript
    display(Javascript(
        'function keepAlive() { '
        'google.colab.kernel.invokeFunction("shell", ["echo alive"], {}); '
        'setTimeout(keepAlive, 120000); '
        '} keepAlive();'
    ))
    print("Keepalive enabled (prevents Colab auto-disconnect)")
except Exception:
    pass

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

# Background removal
print("\nLoading image and removing background...")
raw = Image.open(INPUT_IMAGE)
image = raw.convert("RGBA")
if raw.mode == "RGB":
    image = BackgroundRemover()(image)
print("Background removed.")

# Shape generation
print("\nLoading Hunyuan3D-2mini shape pipeline...")
started = time.time()
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-2mini",
    subfolder="hunyuan3d-dit-v2-mini",
    variant="fp16",
    use_safetensors=True,  # Issue #8: safer + faster weight downloads
)
shape_pipeline.enable_model_cpu_offload()
print(f"Shape pipeline loaded ({time.time() - started:.0f}s)")

print(f"\nGenerating shape (steps={NUM_INFERENCE_STEPS}, chunks={NUM_CHUNKS})...")
mesh = shape_pipeline(
    image=image,
    num_inference_steps=NUM_INFERENCE_STEPS,
    octree_resolution=OCTREE_RESOLUTION,
    num_chunks=NUM_CHUNKS,
    generator=torch.manual_seed(SEED),
    output_type="trimesh",
)[0]

# Issue #11: parameterized filenames
shape_glb = out_dir / f"{AVATAR_NAME}_hunyuan_shape.glb"
mesh.export(shape_glb)
print(f"Shape saved: {shape_glb}")

# Free shape pipeline VRAM before texture
del shape_pipeline
torch.cuda.empty_cache()
gc.collect()
print("Shape pipeline unloaded, VRAM freed.")

# Texture generation
texture_status = "SKIPPED"
final_glb = shape_glb
texture_error = None
try:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline

    # Issue #6: simplify mesh to reduce VRAM for texture on T4
    mesh_for_tex = mesh
    try:
        face_count = len(mesh.faces) if hasattr(mesh, "faces") else 0
        if face_count > 50000:
            print(f"\nMesh has {face_count} faces -- simplifying to reduce VRAM...")
            mesh_for_tex = mesh.simplify_quadric_decimation(face_count=50000)
            print(f"Simplified to {len(mesh_for_tex.faces)} faces.")
    except Exception as simp_err:
        print(f"Mesh simplification skipped: {simp_err}")

    print("\nLoading Hunyuan3D-Paint texture pipeline...")
    paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
    paint_pipeline.enable_model_cpu_offload()

    print("Generating texture...")
    textured_mesh = paint_pipeline(mesh_for_tex, image=image)
    final_glb = out_dir / f"{AVATAR_NAME}_hunyuan_textured.glb"
    textured_mesh.export(final_glb)
    texture_status = "PASS"
    print(f"Textured mesh saved: {final_glb}")

    del paint_pipeline
    torch.cuda.empty_cache()
    gc.collect()

except Exception as exc:
    texture_status = "FAIL"
    texture_error = f"{type(exc).__name__}: {exc}"
    print(f"\nTexture generation failed (shape-only GLB will be used):")
    print(f"   {texture_error}")

elapsed = round(time.time() - started, 2)

# Bundle delivery
manifest = {
    "pipeline": "Hunyuan3D-2mini + Hunyuan3D-Paint",
    "input_image": INPUT_IMAGE,
    "avatar_name": AVATAR_NAME,
    "seed": SEED,
    "octree_resolution": OCTREE_RESOLUTION,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "num_chunks": NUM_CHUNKS,
    "shape_glb": str(shape_glb),
    "final_glb": str(final_glb),
    "texture_status": texture_status,
    "texture_error": texture_error,
    "elapsed_seconds": elapsed,
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

delivery_dir = out_dir / f"{AVATAR_NAME}_hunyuan_delivery"
if delivery_dir.exists():
    shutil.rmtree(delivery_dir)
delivery_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(out_dir / "manifest.json", delivery_dir / "manifest.json")
shutil.copy2(final_glb, delivery_dir / final_glb.name)
if final_glb != shape_glb:
    shutil.copy2(shape_glb, delivery_dir / shape_glb.name)

zip_path = shutil.make_archive(
    str(out_dir / f"{AVATAR_NAME}_hunyuan_delivery"), "zip", root_dir=str(delivery_dir)
)

print(f"\n{'='*60}")
print(f"DONE in {elapsed}s")
print(f"  Shape:   {shape_glb.name}")
print(f"  Texture: {texture_status}")
print(f"  Zip:     {zip_path}")
print(f"{'='*60}")
print(json.dumps(manifest, indent=2))


## 7. Download Delivery Bundle

Downloads the `.zip` containing the `.glb` file(s) and manifest.


In [ ]:
import subprocess
from pathlib import Path
from google.colab import files

out_dir = Path(OUT_DIR)
zip_file = out_dir / f"{AVATAR_NAME}_hunyuan_delivery.zip"

print(f"\nContents of {out_dir}:")
subprocess.run(["ls", "-lah", str(out_dir)], check=False)

if zip_file.exists():
    print(f"\nDownloading {zip_file.name}...")
    files.download(str(zip_file))
else:
    print(f"\nZip not found: {zip_file}")
    print("Available files:")
    for f in out_dir.iterdir():
        print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")


## Summary

### What was generated
- **Shape GLB**: Untextured 3D mesh from Hunyuan3D-2mini
- **Textured GLB**: Painted mesh from Hunyuan3D-Paint (if VRAM allowed)
- **Delivery ZIP**: Bundle with manifest + GLB files

### If texture failed
The T4 has 16 GB VRAM. Texture can OOM on complex meshes. Options:
1. Re-run with SAFE preset: `NUM_INFERENCE_STEPS=24, NUM_CHUNKS=8000`
2. Use the shape-only `.glb` and texture it in Blender
3. Upgrade to Colab Pro (A100 GPU) for reliable texture

### Next steps
- Import the `.glb` into Blender, Unity, or Three.js
- Use Blender UV unwrap + bake for production-quality textures
